# Transform Refund Data
## 1.Extract Specific Portion of string from refunds_reason using split function
## 2.Extract Specific Portion of string from refunds_reason using regex Expression
## 3.Extract time and date from refunds_timestamp
## 4.Transform the data to silver schema foreign catalog

### Extract Specfic portion of string from refunds_reason using split function

In [0]:
df_refunds = spark.read.table("gizmobox_nara.bronze.py_refunds")
display(df_refunds)

In [0]:
%sql
SELECT * FROM asql_gizmobox_nara_catalog_ui.dbo.refundsnara

In [0]:
import pyspark.sql.functions as f
df_refunds_reason_split = (
    df_refunds
     .select("payment_id",
              "refund_amount",
              "refund_id",
              f.split("refund_reason", ":")[0].alias("refund_reason"),
              f.split("refund_reason", ":")[1].alias("refund_source"),
              "refund_timestamp"
              )
)
display(df_refunds_reason_split)

In [0]:
%sql
SELECT refund_id,payment_id,refund_timestamp,refund_amount,
SPLIT(refund_reason,':')[0] AS refund_reason,
SPLIT(refund_reason, ':')[1] AS refund_source
FROM asql_gizmobox_nara_catalog_ui.dbo.refundsnara

# 2.Extract Specific Portion of string from refunds_reason using regex Expression

In [0]:
%sql
SELECT refund_id,payment_id,refund_timestamp,refund_amount,
regexp_extract(refund_reason,'^([^:]+):', 1) AS refund_reason,
regexp_extract(refund_reason,'^[^:]+:(.*)$',1) AS refund_source
FROM asql_gizmobox_nara_catalog_ui.dbo.refundsnara

### Extracting Time and date from refund_timestamp

In [0]:
df_refunds_timestamp = (
    df_refunds_reason_split
    .select("payment_id",
            "refund_amount",
            "refund_id",
            "refund_reason",
            f.date_format("refund_timestamp", "yyyy-MM-dd").alias("refund_date"),
            f.date_format("refund_timestamp", "HH:mm:ss").alias("refund_time")
            )
)
display(df_refunds_timestamp)



In [0]:
df_refund_split_new = (
    df_refunds
    .select("payment_id",
            "refund_amount",
            "refund_id",
            f.date_format("refund_timestamp", "yyyy-MM-dd").alias("refund_date"),
            f.date_format("refund_timestamp", "HH:mm:ss").alias("refund_time"),
            f.regexp_extract("refund_reason", '^([^:]+):', 1).alias("refund_reason"),
            f.regexp_extract("refund_reason", '^[^:]+:(.*)$', 1).alias("refund_source")
    )
)
display(df_refund_split_new)

In [0]:
%sql
SELECT refund_id,payment_id,CAST(date_format(refund_timestamp,'yyyy-MM-dd') AS DATE)  AS refund_date,
date_format(refund_timestamp,'hh:mm:ss') AS refund_time,refund_amount,
regexp_extract(refund_reason,'^([^:]+):', 1) AS refund_reason,
regexp_extract(refund_reason,'^[^:]+:(.*)$',1) AS refund_source
FROM asql_gizmobox_nara_catalog_ui.dbo.refundsnara

### Transfrom to Silver SChema

In [0]:
df_refunds_timestamp.writeTo("gizmobox_nara.silver.py_refunds").createOrReplace()

In [0]:
df = spark.read.table("gizmobox_nara.silver.py_refunds" )
display(df)

In [0]:
%sql
CREATE SCHEMA asql_gizmobox_nara_catalog_ui.silver

In [0]:
%sql
CREATE TABLE gizmobox_nara.silver.refunds
SELECT refund_id,payment_id,CAST(date_format(refund_timestamp,'yyyy-MM-dd') AS DATE)  AS refund_date,
date_format(refund_timestamp,'hh:mm:ss') AS refund_time,refund_amount,
regexp_extract(refund_reason,'^([^:]+):', 1) AS refund_reason,
regexp_extract(refund_reason,'^[^:]+:(.*)$',1) AS refund_source
FROM asql_gizmobox_nara_catalog_ui.dbo.refundsnara

In [0]:
%sql
select * from gizmobox_nara.silver.refunds

In [0]:
%sql DESC EXTENDED gizmobox_nara.silver.refunds